In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 55,
   "id": "bcc44d4f",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import requests\n",
    "import json\n",
    "from bs4 import BeautifulSoup\n",
    "import time\n",
    "from urllib.robotparser import RobotFileParser\n",
    "import argparse\n",
    "from tqdm import tqdm\n",
    "import os\n",
    "from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Date, func, case, select\n",
    "from sqlalchemy.ext.declarative import declarative_base\n",
    "from sqlalchemy.orm import relationship, sessionmaker\n",
    "from datetime import date\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "9cc1e672",
   "metadata": {},
   "source": [
    "2.1.1 Data Format Converter"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "20125ffb",
   "metadata": {},
   "source": [
    "1. Build a Python program that converts data between CSV, JSON, Excel, and Text\n",
    "formats."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "5168734d",
   "metadata": {},
   "outputs": [],
   "source": [
    "class DataConverter:\n",
    "    \n",
    "    def __init__(self, input_path):\n",
    "        self.input_path = input_path\n",
    "        self.df = None\n",
    "        \n",
    "    def read_file(self):\n",
    "        ext = self.input_path.split('.')[-1].lower()\n",
    "        \n",
    "        if ext == 'csv':\n",
    "            self.df = pd.read_csv(self.input_path)\n",
    "            \n",
    "        elif ext == 'json':\n",
    "            with open(self.input_path) as f:\n",
    "                data = json.load(f)\n",
    "                \n",
    "            # Handle nested JSON\n",
    "            if isinstance(data, list):\n",
    "                self.df = pd.json_normalize(data)\n",
    "            else:\n",
    "                self.df = pd.json_normalize([data])\n",
    "                \n",
    "        elif ext in ['xlsx', 'xls']:\n",
    "            self.df = pd.read_excel(self.input_path)\n",
    "            \n",
    "        elif ext == 'txt':\n",
    "            self.df = pd.read_csv(self.input_path, delimiter='\\t')\n",
    "            \n",
    "        else:\n",
    "            raise ValueError(\"Unsupported file format.\")\n",
    "            \n",
    "        print(\"File loaded successfully!\")\n",
    "        return self.df\n",
    "    \n",
    "    \n",
    "    def convert(self, output_path):\n",
    "        ext = output_path.split('.')[-1].lower()\n",
    "        \n",
    "        if ext == 'csv':\n",
    "            self.df.to_csv(output_path, index=False)\n",
    "            \n",
    "        elif ext == 'json':\n",
    "            self.df.to_json(output_path, orient='records', indent=4)\n",
    "            \n",
    "        elif ext in ['xlsx', 'xls']:\n",
    "            self.df.to_excel(output_path, index=False)\n",
    "            \n",
    "        elif ext == 'txt':\n",
    "            self.df.to_csv(output_path, sep='\\t', index=False)\n",
    "            \n",
    "        else:\n",
    "            raise ValueError(\"Unsupported output format.\")\n",
    "            \n",
    "        print(f\"File converted and saved as {output_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6c9d3749",
   "metadata": {},
   "source": [
    "2. How will your program handle nested JSON structures during conversion?"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "81a7f53b",
   "metadata": {},
   "source": [
    "When converting nested JSON:\n",
    "->We use pandas.json_normalize()\n",
    "->It flattens nested structures\n",
    "->Nested keys become column names using dot notation\n",
    "Example:\n",
    "{\n",
    "  \"name\": \"John\",\n",
    "  \"address\": {\n",
    "    \"city\": \"New York\",\n",
    "    \"zip\": 10001\n",
    "  }\n",
    "}\n",
    "becomes:\n",
    "| name | address.city | address.zip |\n",
    "| ---- | ------------ | ----------- |\n",
    "|john  |new york      |10001        |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "c1312e43",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>name</th>\n",
       "      <th>contact.email</th>\n",
       "      <th>contact.phone</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Alice</td>\n",
       "      <td>alice@gmail.com</td>\n",
       "      <td>123456</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "    name    contact.email contact.phone\n",
       "0  Alice  alice@gmail.com        123456"
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "nested_json = {\n",
    "    \"name\": \"Alice\",\n",
    "    \"contact\": {\n",
    "        \"email\": \"alice@gmail.com\",\n",
    "        \"phone\": \"123456\"\n",
    "    }\n",
    "}\n",
    "\n",
    "df_nested = pd.json_normalize(nested_json)\n",
    "df_nested"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3742bb17",
   "metadata": {},
   "source": [
    "3. How do you validate data types and detect missing values during conversion?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "0c784860",
   "metadata": {},
   "outputs": [],
   "source": [
    "def validate_data(df):\n",
    "    \n",
    "    print(\"----- DATA TYPES -----\")\n",
    "    print(df.dtypes)\n",
    "    \n",
    "    print(\"\\n----- MISSING VALUES -----\")\n",
    "    print(df.isnull().sum())\n",
    "    \n",
    "    print(\"\\n----- DUPLICATE ROWS -----\")\n",
    "    print(df.duplicated().sum())\n",
    "    \n",
    "    print(\"\\n----- SUMMARY STATISTICS -----\")\n",
    "    print(df.describe(include='all'))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b12dd26d",
   "metadata": {},
   "source": [
    "program validates data by:\n",
    "1. Checking column data types using:\n",
    "df.dtypes\n",
    "2. Detecting missing values using:\n",
    "df.isnull().sum()\n",
    "3. Detecting duplicates:\n",
    "df.duplicated().sum()\n",
    "4. Identifying inconsistencies via:\n",
    "    ->Mixed data types\n",
    "    ->Unexpected null values\n",
    "    ->Outliers in numerical columns"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "07bb2d07",
   "metadata": {},
   "source": [
    "5. Generate a data quality report showing missing values, data types, and inconsistencies."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "e0daf913",
   "metadata": {},
   "outputs": [],
   "source": [
    "def generate_quality_report(df):\n",
    "    \n",
    "    report = pd.DataFrame()\n",
    "    \n",
    "    report['Data Type'] = df.dtypes\n",
    "    report['Missing Values'] = df.isnull().sum()\n",
    "    report['Missing %'] = (df.isnull().sum() / len(df)) * 100\n",
    "    report['Unique Values'] = df.nunique()\n",
    "    \n",
    "    report['Sample Value'] = df.iloc[0]\n",
    "    \n",
    "    return report"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "97e15bf1",
   "metadata": {},
   "source": [
    "4. Design a command-line interface (CLI) for selecting input and output formats."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6ab0be57",
   "metadata": {},
   "outputs": [],
   "source": [
    "def main():\n",
    "    parser = argparse.ArgumentParser(description=\"Data Format Converter\")\n",
    "    \n",
    "    parser.add_argument(\"input\", help=\"Input file path\")\n",
    "    parser.add_argument(\"output\", help=\"Output file path\")\n",
    "    \n",
    "    args = parser.parse_args()\n",
    "    \n",
    "    converter = DataConverter(args.input)\n",
    "    df = converter.read_file()\n",
    "    \n",
    "    validate_data(df)\n",
    "    \n",
    "    report = generate_quality_report(df)\n",
    "    print(\"\\nData Quality Report:\\n\", report)\n",
    "    \n",
    "    converter.convert(args.output)\n",
    "main()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c4fc8c31",
   "metadata": {},
   "source": [
    "2.2.1 Student Management System"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "634d892a",
   "metadata": {},
   "source": [
    "6. Design a Relational Database Schema"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "7e536176",
   "metadata": {},
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "C:\\Users\\milim\\AppData\\Local\\Temp\\ipykernel_6728\\2483821873.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)\n",
      "  Base = declarative_base()\n"
     ]
    }
   ],
   "source": [
    "Base = declarative_base()\n",
    "\n",
    "# Students table\n",
    "class Student(Base):\n",
    "    __tablename__ = 'students'\n",
    "    id = Column(Integer, primary_key=True)\n",
    "    name = Column(String, nullable=False)\n",
    "    email = Column(String, unique=True, nullable=False)\n",
    "    enrollments = relationship(\"Enrollment\", back_populates=\"student\")\n",
    "    attendance_records = relationship(\"Attendance\", back_populates=\"student\")\n",
    "\n",
    "# Courses table\n",
    "class Course(Base):\n",
    "    __tablename__ = 'courses'\n",
    "    id = Column(Integer, primary_key=True)\n",
    "    name = Column(String, nullable=False)\n",
    "    credits = Column(Integer, nullable=False)\n",
    "    enrollments = relationship(\"Enrollment\", back_populates=\"course\")\n",
    "    attendance_records = relationship(\"Attendance\", back_populates=\"course\")\n",
    "\n",
    "# Enrollments table\n",
    "class Enrollment(Base):\n",
    "    __tablename__ = 'enrollments'\n",
    "    id = Column(Integer, primary_key=True)\n",
    "    student_id = Column(Integer, ForeignKey('students.id'))\n",
    "    course_id = Column(Integer, ForeignKey('courses.id'))\n",
    "    grade = Column(Float)  # Assuming 0.0 - 4.0 scale\n",
    "    student = relationship(\"Student\", back_populates=\"enrollments\")\n",
    "    course = relationship(\"Course\", back_populates=\"enrollments\")\n",
    "\n",
    "# Attendance table\n",
    "class Attendance(Base):\n",
    "    __tablename__ = 'attendance'\n",
    "    id = Column(Integer, primary_key=True)\n",
    "    student_id = Column(Integer, ForeignKey('students.id'))\n",
    "    course_id = Column(Integer, ForeignKey('courses.id'))\n",
    "    date = Column(Date)\n",
    "    status = Column(String)  # e.g., \"Present\", \"Absent\"\n",
    "    student = relationship(\"Student\", back_populates=\"attendance_records\")\n",
    "    course = relationship(\"Course\", back_populates=\"attendance_records\")\n",
    "\n",
    "# Create the SQLite database\n",
    "engine = create_engine('sqlite:///school.db')\n",
    "Base.metadata.create_all(engine)\n",
    "\n",
    "# Create a session\n",
    "Session = sessionmaker(bind=engine)\n",
    "session = Session()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "310f050b",
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Add students ---\n",
    "students = [\n",
    "    Student(name=\"Alice Johnson\", email=\"alice@example.com\"),\n",
    "    Student(name=\"Bob Smith\", email=\"bob@example.com\"),\n",
    "    Student(name=\"Charlie Lee\", email=\"charlie@example.com\")\n",
    "]\n",
    "\n",
    "session.add_all(students)\n",
    "session.commit()\n",
    "\n",
    "# --- Add courses ---\n",
    "courses = [\n",
    "    Course(name=\"Mathematics\", credits=3),\n",
    "    Course(name=\"Physics\", credits=4),\n",
    "    Course(name=\"History\", credits=2)\n",
    "]\n",
    "\n",
    "session.add_all(courses)\n",
    "session.commit()\n",
    "\n",
    "# --- Add enrollments with grades ---\n",
    "enrollments = [\n",
    "    Enrollment(student_id=1, course_id=1, grade=3.5),\n",
    "    Enrollment(student_id=1, course_id=2, grade=3.0),\n",
    "    Enrollment(student_id=1, course_id=3, grade=4.0),\n",
    "    \n",
    "    Enrollment(student_id=2, course_id=1, grade=2.0),\n",
    "    Enrollment(student_id=2, course_id=2, grade=2.5),\n",
    "    Enrollment(student_id=2, course_id=3, grade=1.5),\n",
    "    \n",
    "    Enrollment(student_id=3, course_id=1, grade=4.0),\n",
    "    Enrollment(student_id=3, course_id=2, grade=3.8),\n",
    "    Enrollment(student_id=3, course_id=3, grade=3.9),\n",
    "]\n",
    "\n",
    "session.add_all(enrollments)\n",
    "session.commit()\n",
    "\n",
    "# --- Add attendance records ---\n",
    "attendance_records = [\n",
    "    # Alice\n",
    "    Attendance(student_id=1, course_id=1, date=date(2026,2,1), status=\"Present\"),\n",
    "    Attendance(student_id=1, course_id=1, date=date(2026,2,2), status=\"Absent\"),\n",
    "    Attendance(student_id=1, course_id=2, date=date(2026,2,1), status=\"Present\"),\n",
    "    Attendance(student_id=1, course_id=3, date=date(2026,2,1), status=\"Present\"),\n",
    "    \n",
    "    # Bob\n",
    "    Attendance(student_id=2, course_id=1, date=date(2026,2,1), status=\"Absent\"),\n",
    "    Attendance(student_id=2, course_id=2, date=date(2026,2,1), status=\"Present\"),\n",
    "    Attendance(student_id=2, course_id=3, date=date(2026,2,1), status=\"Absent\"),\n",
    "    \n",
    "    # Charlie\n",
    "    Attendance(student_id=3, course_id=1, date=date(2026,2,1), status=\"Present\"),\n",
    "    Attendance(student_id=3, course_id=2, date=date(2026,2,1), status=\"Present\"),\n",
    "    Attendance(student_id=3, course_id=3, date=date(2026,2,1), status=\"Present\"),\n",
    "]\n",
    "\n",
    "session.add_all(attendance_records)\n",
    "session.commit()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "000d6476",
   "metadata": {},
   "source": [
    "7. Write SQL queries to calculate the GPA of each student."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "b6ffd924",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Alice Johnson (ID: 1) - GPA: 3.39\n",
      "Bob Smith (ID: 2) - GPA: 2.11\n",
      "Charlie Lee (ID: 3) - GPA: 3.89\n"
     ]
    }
   ],
   "source": [
    "gpa_query = session.query(\n",
    "    Student.id,\n",
    "    Student.name,\n",
    "    (func.sum(Enrollment.grade * Course.credits) / func.sum(Course.credits)).label('GPA')\n",
    ").select_from(Student).join(Enrollment, Student.id == Enrollment.student_id)\\\n",
    "  .join(Course, Enrollment.course_id == Course.id)\\\n",
    "  .group_by(Student.id)\n",
    "\n",
    "for student_id, student_name, gpa in gpa_query:\n",
    "    print(f\"{student_name} (ID: {student_id}) - GPA: {gpa:.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d2d1f420",
   "metadata": {},
   "source": [
    "8. Generate attendance reports for individual students and courses."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "id": "1ba90fd8",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "History - 2026-02-01 - Present\n",
      "Mathematics - 2026-02-01 - Present\n",
      "Mathematics - 2026-02-02 - Absent\n",
      "Physics - 2026-02-01 - Present\n",
      "Alice Johnson - 2026-02-01 - Present\n",
      "Alice Johnson - 2026-02-02 - Absent\n",
      "Bob Smith - 2026-02-01 - Absent\n",
      "Charlie Lee - 2026-02-01 - Present\n"
     ]
    }
   ],
   "source": [
    "#a) Attendance report for a specific student\n",
    "student_id = 1  # Example student\n",
    "\n",
    "attendance_report = session.query(\n",
    "    Course.name,\n",
    "    Attendance.date,\n",
    "    Attendance.status\n",
    ").join(Course).filter(Attendance.student_id == student_id).order_by(Course.name, Attendance.date)\n",
    "\n",
    "for course_name, date, status in attendance_report:\n",
    "    print(f\"{course_name} - {date} - {status}\")\n",
    "\n",
    "#b) Attendance report for a specific course\n",
    "course_id = 1  # Example course\n",
    "\n",
    "attendance_report = session.query(\n",
    "    Student.name,\n",
    "    Attendance.date,\n",
    "    Attendance.status\n",
    ").join(Student).filter(Attendance.course_id == course_id).order_by(Student.name, Attendance.date)\n",
    "\n",
    "for student_name, date, status in attendance_report:\n",
    "    print(f\"{student_name} - {date} - {status}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "696c9e49",
   "metadata": {},
   "source": [
    "9. Analyze course performance using enrollment and grade data."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "id": "2b8ef299",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Mathematics - Average Grade: 3.17\n",
      "Physics - Average Grade: 3.10\n",
      "History - Average Grade: 3.13\n"
     ]
    }
   ],
   "source": [
    "course_performance = session.query(\n",
    "    Course.name,\n",
    "    func.avg(Enrollment.grade).label(\"average_grade\")\n",
    ").join(Enrollment).group_by(Course.id)\n",
    "\n",
    "for course_name, avg_grade in course_performance:\n",
    "    print(f\"{course_name} - Average Grade: {avg_grade:.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c3939b76",
   "metadata": {},
   "source": [
    "10. Identify at-risk students based on grades and attendance patterns."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 27,
   "id": "76277dc8",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "At-Risk: Bob Smith - GPA: 2.11, Attendance: 33.3%\n"
     ]
    }
   ],
   "source": [
    "# --- Attendance percentage per student ---\n",
    "attendance_stats = session.query(\n",
    "    Student.id,\n",
    "    Student.name,\n",
    "    (func.sum(case((Attendance.status=='Present', 1), else_=0)) * 1.0 / func.count(Attendance.id)).label('attendance_rate')\n",
    ").select_from(Student).join(Attendance, Student.id == Attendance.student_id)\\\n",
    "  .group_by(Student.id).all()\n",
    "\n",
    "# --- GPA per student ---\n",
    "gpa_stats = session.query(\n",
    "    Student.id,\n",
    "    (func.sum(Enrollment.grade * Course.credits) / func.sum(Course.credits)).label('GPA')\n",
    ").select_from(Student).join(Enrollment, Student.id == Enrollment.student_id)\\\n",
    "  .join(Course, Enrollment.course_id == Course.id)\\\n",
    "  .group_by(Student.id).all()\n",
    "\n",
    "# --- Combine GPA and attendance to identify at-risk students ---\n",
    "gpa_dict = {s.id: s.GPA for s in gpa_stats}\n",
    "at_risk_students = []\n",
    "\n",
    "for s in attendance_stats:\n",
    "    gpa = gpa_dict.get(s.id, 0)\n",
    "    if gpa < 2.0 or s.attendance_rate < 0.75:  # GPA < 2.0 or attendance < 75%\n",
    "        at_risk_students.append((s.name, gpa, s.attendance_rate))\n",
    "\n",
    "# --- Print at-risk students ---\n",
    "for name, gpa, attendance in at_risk_students:\n",
    "    print(f\"At-Risk: {name} - GPA: {gpa:.2f}, Attendance: {attendance*100:.1f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "78ae08c2",
   "metadata": {},
   "source": [
    "2.3.1Accessing and processing data from APIs (REST, SOAP)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "31a631c7",
   "metadata": {},
   "source": [
    "11. Fetch weather data from at least two different weather APIs."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0853955b",
   "metadata": {},
   "outputs": [],
   "source": [
    "OWM_API_KEY = \"your_openweathermap_api_key\"\n",
    "WA_API_KEY = \"your_weatherapi_key\"#dont have an api key\n",
    "\n",
    "city = \"kathmandu\"\n",
    "\n",
    "# OpenWeatherMap\n",
    "owm_url = f\"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OWM_API_KEY}&units=metric\"\n",
    "owm_response = requests.get(owm_url)\n",
    "owm_data = owm_response.json()\n",
    "print(\"OpenWeatherMap:\", owm_data)\n",
    "\n",
    "# WeatherAPI\n",
    "wa_url = f\"http://api.weatherapi.com/v1/current.json?key={WA_API_KEY}&q={city}&aqi=no\"\n",
    "wa_response = requests.get(wa_url)\n",
    "wa_data = wa_response.json()\n",
    "print(\"WeatherAPI:\", wa_data)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "76aab9c3",
   "metadata": {},
   "source": [
    "12. How do you securely manage API keys in your application?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "05966753",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Store keys in environment variables\n",
    "OWM_API_KEY = os.getenv(\"OWM_API_KEY\")\n",
    "WA_API_KEY = os.getenv(\"WA_API_KEY\")\n",
    "#OR\n",
    "from dotenv import load_dotenv\n",
    "load_dotenv()\n",
    "\n",
    "OWM_API_KEY = os.getenv(\"OWM_API_KEY\")\n",
    "WA_API_KEY = os.getenv(\"WA_API_KEY\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f2fd09f0",
   "metadata": {},
   "source": [
    "13. Handle API rate limits and failed requests gracefully."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6eec9be3",
   "metadata": {},
   "outputs": [],
   "source": [
    "def fetch_json(url, retries=3):\n",
    "    for attempt in range(retries):\n",
    "        response = requests.get(url)\n",
    "        if response.status_code == 200:\n",
    "            return response.json()\n",
    "        elif response.status_code == 429:  # Rate limit exceeded\n",
    "            print(\"Rate limit hit, retrying...\")\n",
    "            import time; time.sleep(2 * (attempt + 1))\n",
    "        else:\n",
    "            print(f\"Request failed: {response.status_code}\")\n",
    "    return None\n",
    "\n",
    "owm_data = fetch_json(owm_url)\n",
    "wa_data = fetch_json(wa_url)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "44f85ef4",
   "metadata": {},
   "source": [
    "14. Normalize weather data obtained from different APIs into a common format."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4d771498",
   "metadata": {},
   "outputs": [],
   "source": [
    "def normalize_weather(owm, wa):\n",
    "    normalized = []\n",
    "\n",
    "    # OpenWeatherMap\n",
    "    if owm:\n",
    "        normalized.append({\n",
    "            \"source\": \"OpenWeatherMap\",\n",
    "            \"temp_c\": owm['main']['temp'],\n",
    "            \"humidity\": owm['main']['humidity'],\n",
    "            \"condition\": owm['weather'][0]['description']\n",
    "        })\n",
    "\n",
    "    # WeatherAPI\n",
    "    if wa:\n",
    "        normalized.append({\n",
    "            \"source\": \"WeatherAPI\",\n",
    "            \"temp_c\": wa['current']['temp_c'],\n",
    "            \"humidity\": wa['current']['humidity'],\n",
    "            \"condition\": wa['current']['condition']['text']\n",
    "        })\n",
    "    return normalized\n",
    "\n",
    "weather_normalized = normalize_weather(owm_data, wa_data)\n",
    "weather_normalized"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7c623222",
   "metadata": {},
   "source": [
    "15. Compare daily weather reports and forecasts from multiple sources."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b2ff869a",
   "metadata": {},
   "outputs": [],
   "source": [
    "for w in weather_normalized:\n",
    "    print(f\"{w['source']}: {w['temp_c']}°C, {w['humidity']}% humidity, {w['condition']}\")\n",
    "\n",
    "# Simple comparison\n",
    "if len(weather_normalized) == 2:\n",
    "    diff = abs(weather_normalized[0]['temp_c'] - weather_normalized[1]['temp_c'])\n",
    "    print(f\"Temperature difference between sources: {diff:.1f}°C\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a8879eb7",
   "metadata": {},
   "source": [
    "16. Implement a basic alert system based on weather conditions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "46116265",
   "metadata": {},
   "outputs": [],
   "source": [
    "alerts = []\n",
    "for w in weather_normalized:\n",
    "    if w['temp_c'] > 30:\n",
    "        alerts.append(f\"{w['source']} alert: High temperature {w['temp_c']}°C\")\n",
    "    if 'rain' in w['condition'].lower():\n",
    "        alerts.append(f\"{w['source']} alert: Rain expected ({w['condition']})\")\n",
    "\n",
    "for alert in alerts:\n",
    "    print(alert)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1da235d0",
   "metadata": {},
   "source": [
    "2.4.1 Web scraping using requests and BeautifulSoup"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c8320f97",
   "metadata": {},
   "source": [
    "17. Scrape news articles from multiple websites while following ethical scraping practices."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 40,
   "id": "4647878a",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "BBC Headlines:\n",
      "- Trump's theatrical State of the Union address offers little hint of any change in course\n",
      "- Fact-checking Trump's longest ever State of the Union\n",
      "- BBC sees damage in Puerto Vallarta after Mexican cartel violence\n",
      "- King asked to save UK's oldest Indian restaurant\n",
      "- Japan to deploy missiles on island near Taiwan by 2031\n",
      "- Don't break up NewJeans and I'll forgo $18m payout, says ex-K-pop boss\n",
      "- Chinese dance group's tour triggers bomb threat against Australian PM\n",
      "- Modi's Israel visit to test India's priorities in the Middle East\n",
      "- Threat of further violence looms after Mexican cartel rampage\n",
      "- German chancellor lands in Beijing for inaugural China trip\n"
     ]
    }
   ],
   "source": [
    "def is_allowed(url, user_agent=\"*\"):\n",
    "    base_url = \"/\".join(url.split(\"/\")[:3])\n",
    "    robots_url = base_url + \"/robots.txt\"\n",
    "    \n",
    "    rp = RobotFileParser()\n",
    "    rp.set_url(robots_url)\n",
    "    rp.read()\n",
    "    \n",
    "    return rp.can_fetch(user_agent, url)\n",
    "#scrape bbc headlines\n",
    "bbc_url = \"https://www.bbc.com/news\"\n",
    "\n",
    "if is_allowed(bbc_url):\n",
    "    headers = {\"User-Agent\": \"Mozilla/5.0\"}\n",
    "    response = requests.get(bbc_url, headers=headers)\n",
    "    soup = BeautifulSoup(response.text, \"html.parser\")\n",
    "\n",
    "    headlines = []\n",
    "    for h in soup.find_all(\"h2\")[:10]:\n",
    "        headlines.append(h.get_text(strip=True))\n",
    "\n",
    "    print(\"BBC Headlines:\")\n",
    "    for h in headlines:\n",
    "        print(\"-\", h)\n",
    "else:\n",
    "    print(\"Scraping not allowed by robots.txt\")\n",
    "\n",
    "time.sleep(2)  # polite delay"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8677956d",
   "metadata": {},
   "source": [
    "18. How do you ensure compliance with robots.txt and terms of service?\n",
    "we can ensure compliance by:\n",
    "1. using RobotFileParser to check permission.\n",
    "2. reading website robots.txt manually.\n",
    "3. not scrapping:\n",
    "    ->login proctected page\n",
    "    ->paid content.\n",
    "4. adding delays\n",
    "5.setting a proper user-agent header.\n",
    "6. not overlaoding servers."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4f28fa5b",
   "metadata": {},
   "source": [
    "19. Extract headlines, full content, authors, publication dates, and categories from news\n",
    "pages."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 50,
   "id": "3990e8c2",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "{'title': None, 'author': 'पिपुल्स न्युज', 'date': 'February 24, 2026', 'category': None, 'content': '  People’s News : चुनाव आउन जम्मा नौ दिन बाँकी रहँदा मुस्ताङ … Read more   People’s News : सार्वजनिक पदको दुरुपयोग गरेको आशङ्कामा सोमबार पक्राउ परेको … Read more   People’s News Kathmandu : सैन्य शासनको प्रयास र विद्रोहको योजना बनाएको … Read more   People’s News : सर्वोच्च अदालतले सन् १९७७ मा लागू भएको ‘इन्टर्नेसनल … Read more   People’s News प्रतिबन्धित लागुऔषधvसहित दुईजनालाई पक्राउ सिराहाको सीमावर्ती भगवानपुर गाउँपालिका– ३ … Read more   People’s News kathmandu : आगामी फागुन २१ गते हुने प्रतिनिधिसभा निर्वाचन … Read more   People’s News Kathmandu : राष्ट्रिय स्वतन्त्र पार्टी (रास्वपा) का वरिष्ठ नेता … Read more   People’s News : घरघरमा गेरु रङ्गको झण्डा, केहीमा हनुमान्\\u200cको तस्बिर । … Read more   People’s News : कृत्रिम बौद्धिकता (एआई) ले परम्परागत गहना उद्योगमा नयाँ … Read more   People’s News : टी-२० विश्वकपको ३६ औं खेल आज श्रीलंका र … Read more   People’s News kathmandu :\\xa0राष्ट्रिय स्वतन्त्र पार्टी (रास्वपा) का सभापति रवि लामिछानेले … Read more   पूर्वराजा शाहले देश अहिले इतिहासकै दर्दनाक अवस्थामा आइपुगेको दाबी गरेका हुन् … Read more   People’s News : ५० सदस्यीय मन्त्रिपरिषद्को घोषणा सहित ७० बर्षिय बृद्धले … Read more   People’s News काठमाडौं :\\xa0वातावरण विभागको तथ्यांकले सोमबार र मंगलबार इलाम सबैभन्दा … Read more   पिपुल्स न्युज काठमाडौँ: हालका दिनहरूमा युवाहरूमा कुनै खास व्यक्ति, राजनीतिक दल वा विचारप्रति ‘अन्धभक्ति’ बढ्दै गएको देखिन्छ। आलोचनात्मक चेतनाभन्दा बढी भावनात्मक संलग्नता र तर्कहीन समर्थनको यो प्रवृत्तिले समाजमा गम्भीर बहस सिर्जना गरेको छ। आखिर किन युवा पुस्ता यो बाटोमा अग्रसर भइरहेको छ त? यसका पछाडि केही प्रमुख कारणहरू केलाउन सकिन्छ। मुख्य कारणहरू : डिजिटल र सामाजिक सञ्जालको प्रभाव : सामाजिक सञ्जाल अहिले सूचना र विचार आदानप्रदानको मुख्य माध्यम बनेको छ। एल्गोरिदमले प्रयोगकर्तालाई उस्तै विचार भएका मानिसहरूसँग जोड्दा ‘इको चेम्बर’ (प्रतिध्वनि कक्ष) सिर्जना हुन्छ। यसले फरक मत सुन्ने अवसर घटाउँछ र आफ्नै विचारलाई मात्र सत्य मान्ने प्रवृत्ति बढाउँछ। साथै, गलत सूचना र उत्तेजित सामग्री सहजै फैलिँदा युवाहरूलाई त्यसको पछि लाग्न प्रेरित गर्छ। पहिचान र आबद्धताको खोजी : युवावस्था पहिचान निर्माणको महत्वपूर्ण चरण हो। कतिपय युवाहरूले कुनै खास समूह, नेता वा विचारधारामा आफूलाई आबद्ध गराएर सुरक्षित महसुस गर्छन् र आफ्नो अस्तित्व खोज्छन्। यसले उनीहरूलाई एक्लोपनबाट बचाई एक ‘सामूहिक पहिचान’ प्रदान गर्छ, जसप्रति उनीहरू सहजै भावनात्मक रूपमा जोडिन पुग्छन्। आर्थिक र सामाजिक निराशा : बेरोजगारी, भ्रष्टाचार, र राजनीतिक अस्थिरता जस्ता समस्याहरूले युवाहरूमा निराशा र आक्रोश उत्पन्न गर्छ। यस्तो अवस्थामा कुनै बलियो नेता वा विचारधाराले छिटो समाधानका सपना देखाउँदा वा वर्तमान समस्याहरूको लागि कुनै एक पक्षलाई दोषी ठहर्\\u200dयाउँदा युवाहरू त्यसप्रति आकर्षित हुन सक्छन्। उनीहरूले वर्तमान व्यवस्थाबाट मुक्ति खोज्दै विकल्पको रूपमा ‘अन्धभक्ति’ लाई अपनाउन सक्छन्। केही नेताहरूमा जनमानसलाई भावनात्मक रूपमा जोड्न सक्ने विशेष क्षमता हुन्छ। उनीहरूको भाषण शैली, व्यक्तित्व र दृढ संकल्पले युवाहरूलाई प्रभावित गर्छ। तर्कभन्दा बढी भावनामा आधारित यस्तो आकर्षणले आलोचनात्मक सोचलाई ओझेलमा पारी निर्विवाद समर्थनका लागि प्रेरित गर्छ। आलोचनात्मक सोचको कमी र मिडिया साक्षरताको अभाव: विद्यालय तथा विश्वविद्यालय स्तरमा आलोचनात्मक सोच (critical thinking) र मिडिया साक्षरताको पर्याप्त अभ्यास नहुँदा युवाहरूलाई सही र गलत सूचना छुट्याउन गाह्रो हुन्छ। यसले उनीहरूलाई कुनै पनि प्रोपोगान्डा (प्रचार) को शिकार बन्न सजिलो बनाउँछ, जसले अन्धभक्तिलाई बढावा दिन्छ\\xa0 युवामा बढ्दो ‘अन्धभक्ति’ को प्रवृत्ति समाज, शिक्षा प्रणाली र राजनीतिक नेतृत्व सबैका लागि चिन्ताको विषय हो। स्वस्थ लोकतन्त्रका लागि आलोचनात्मक चेतना, बहुलवादको सम्मान र तथ्यमा आधारित विचार अपरिहार्य हुन्छ। यस चुनौतीको सामना गर्न मिडिया साक्षरता अभिवृद्धि, बहसको संस्कृति प्रवर्धन र युवाहरूलाई अर्थपूर्ण संलग्नताका अवसरहरू प्रदान गर्नु आवश्यक छ Read more   People’s News Kathmandu : राष्ट्रिय सहकारी नियमन प्राधिकरणको व्यवहारले ‘बैंकिङ माफियाको … Read more   People’s News Kathmandu : कुनै निकाय एक्कासी गठन भई नियम लागु … Read more   पिपुल्स न्युज : काठमाडौँ // नेकपा एमालेका अध्यक्ष तथा पूर्वप्रधानमन्त्री केपी … Read more   पिपुल्स न्युज काठमाडौँ : कालोसुची गरे पछि ऋणीले ऋण चुक्ता गर्छ … Read more   पिपुल्स न्युज काठमाडौँ — पछिल्ला वर्षहरूमा नेपाली राजनीतिमा विदेशी हस्तक्षेपको विषय … Read more   पिपुल्स न्युज काठमाडौँ : लुसिफरको छायाँमा देश : विवेक हराउँदा बढ्दै … Read more   Peoples News : जेन जी आन्दोलनका बेला भएका प्रदर्शन मात्र होइन … Read more   People’s News : नेपालका विभिन्न स्थानमा हिमपात र हिउँदे वर्षा हुँदा … Read more   पिपुल्स न्युज काठमाडौँ : सहकारी क्षेत्रलाई केवल बचत–ऋणको माध्यम मात्र ठान्ने … Read more   पिपुल्स न्युज काठमाडौं । जनतामा गहिरिँदै गएको गुलाम मानसिकता नै आज … Read more   पिपुल्स न्युज : निर्वाचन आयोगका अनुसार समानुपतिक सूचीमा नाम रहेका व्यक्तिहरूले … Read more   पिपुल्स न्युज काठमाडौँ : नेपाल आज स्वतन्त्र राष्ट्रजस्तो होइन, भ्रष्ट शासकहरूको … Read more   पिपुल्स न्युज काठमाडौँ //प्रधानमन्त्रीसँग कुलमानको बार्गेनिङ, भने- म हट्नु परे मेरै … Read more   People’s News //काठमाडौं// आजका युवाको हातमा किताब होइन, मोबाइल छ । … Read more   People’s News : काठमाडौं // आजको युवा पुस्ता परिवर्तनको इन्जिन होइन, … Read more   People’s News : सत्ता, अहंकार र षड्यन्त्रको खेलमा जनता बलिदान बन्दै … Read more   Peoples News : धार्मिक सङ्कटतर्फ समाज : आस्थाको नाममा विभाजन, राजनीति … Read more   people’s News : काठमाडौँ । कुनै पनि देशको वास्तविक शक्ति उसका जनता हुन् । तर जब जनता सचेत नागरिक होइन, शासकको आदेश मान्ने गुलाम बन्छन्, त्यहीँबाट राष्ट्र संकटतर्फ धकेलिन थाल्छ । जनताको मौनता, डर र अन्धविश्वासले निरंकुशता जन्माउँछ र निरंकुशताले देशलाई कमजोर बनाउँछ । आजको सन्दर्भमा पनि प्रश्न गम्भीर बनेको छ । हामी नागरिक हौँ कि केवल मतदाता ? चुनावका बेला प्रयोग हुने र पछि बिर्सिइने भीड ? यदि जनता आफ्ना अधिकार, कर्तव्य र राजनीतिक चेतनाप्रति उदासीन भए, सत्ता केही व्यक्तिको हातमा केन्द्रित हुन्छ। त्यहीँबाट भ्रष्टाचार, अन्याय र दमन मौलाउँछ ।\\xa0 चेतनाको अभाव, गुलामीको सुरुवात :\\xa0जनता गुलाम बन्ने प्रक्रिया एकैदिन सुरु हुँदैन । गलतलाई गलत भन्न नसक्नु, नेताको आलोचना गर्न डराउनु, प्रश्न उठाउनेलाई देशद्रोही ठान्नु यिनै प्रवृत्तिले गुलामीको जरा गाड्छ । जब नागरिकले विवेक गुमाउँछन्, सत्ता स्वेच्छाचारी बन्छ ।\\xa0 युवाको भूमिका निर्णायक :\\xa0देशलाई संकटबाट जोगाउने कि धकेल्ने – यो आजको युवाको चेतनामा निर्भर छ । युवाहरू यदि व्यक्ति–पूजामा अल्झिए, सामाजिक सञ्जालको नारामा सीमित भए र यथार्थ राजनीतिबाट टाढा भए भने राष्ट्र झन् कमजोर हुन्छ । तर युवाले प्रश्न गरे, तथ्य खोजे र अन्यायविरुद्ध आवाज उठाए भने परिवर्तन सम्भव छ । लोकतन्त्र सजावट होइन, जिम्मेवारी हो :\\xa0लोकतन्त्र केवल संविधानको पाना वा भाषणको शब्द होइन । यो जनताको निरन्तर निगरानी, सहभागिता र प्रतिरोधबाट जीवित रहन्छ । जनता चुप लागे भने लोकतन्त्र खोक्रो हुन्छ र देश संकटमा फस्छ ।\\xa0\\xa0 जनता गुलाम हुँदा देश बाँच्दैन : देश बाहिरबाट होइन, भित्रैबाट सकिन्छ ।\\xa0 जब जनता शासकको गलत निर्णयमाथि प्रश्न गर्न छोड्छ, तब देश संकटमा प्रवेश गरिसकेको हुन्छ । जनता सचेत नागरिक होइन, तालि बजाउने भीड बनेपछि सत्ता निरंकुश हुन्छ र राष्ट्र कमजोर ।\\xa0आज जनता बोल्न डराउँछन्, प्रश्न गर्न हिच्किचाउँछन् । नेताको गलत काम देख्दा पनि “हाम्रो मान्छे हो” भनेर चुप लाग्छन् । यही मौनताले देशलाई खोक्रो बनाइरहेको छ । जनताको चुप्पी नै शासकको सबैभन्दा ठूलो हतियार बनेको छ ।\\xa0 गुलामी आधुनिक रूपमै आएको छ :\\xa0आजको गुलामी हत्कडी र कोर्राले होइन, नारा, भावुक भाषण र झुटा राष्ट्रवादले आएको छ । सोच्न बन्द गराइन्छ, प्रश्न उठाउन देशद्रोहको आरोप लगाइन्छ । नागरिकलाई चेतनशील होइन, आज्ञाकारी उपभोक्ता बनाइँदैछ ।\\xa0 युवा कि अन्धभक्त\\xa0 ? :\\xa0सबैभन्दा खतरनाक कुरा युवा चेतनशील होइन, अन्धभक्त बन्दैछन् । नेताको फोटो प्रोफाइलमा राखेर देश बदलिन्छ भन्ने भ्रम फैलिएको छ । युवा प्रश्न सोध्दैन, तर्क गर्दैन सिर्फ बचाउ गर्छ । यस्तो युवाबाट परिवर्तन होइन, अधिनायकवाद जन्मिन्छ ।\\xa0 जनता कमजोर हुँदा नेता शक्तिशाली हुन्छ : जब जनता डराउँछ, सत्ता ढुक्क हुन्छ । जब जनता मौन बस्छ, भ्रष्टाचार मौलाउँछ । जब जनता झुक्छ, देश लुटिन्छ । अब निर्णय जनताकै हातमा छ – चुप लाग्ने कि बोल्ने ? पूजा गर्ने कि प्रश्न गर्ने ? भीड बन्ने कि नागरिक हुने ?\\xa0   People’s News : नेपालको भविष्य आज निर्णायक मोडमा छ । यो … Read more   People’s News\\xa0 Kathmandu : अमेरिकी साम्राज्यवादको काहारमा नेपाल\\xa0काठमाडौं । विश्व राजनीतिमा … Read more   पिपुल्स न्युज : काठमाडौं — राजनीतिमा नयाँपन र परिवर्तनको दाबी गर्दै … Read more   पिपुल्स न्युज काठमाण्डौ : तपाईंको चेक बाउन्स भयो भने अब के … Read more   Peoples news : नेपाली राजनीति आज गम्भीर मोडमा उभिएको छ। सरकार … Read more   Peoples News : तथ्यांकअनुसार डिसेम्बरमा आठ हजार १६५ व्यक्ति डेंगुबाट संक्रमित … Read more   Peoples News :\\xa0काठमाडौं, १४ पुस । एकै दिन नौजना फरार अभियुक्तहरु … Read more   Peoples News : काठमाडौं, १४ पुस । न्युजिल्यान्ड पठाइदिने भन्दै ठगीगर्ने … Read more   Peoples News : काठमाडौं, १४ पुस । निर्वाचन आयोगले राष्ट्रियसभा सदस्य … Read more   Peoples News : काठमाडौं, पुस । आज समानुपातिक निर्वाचन प्रणालीतर्फको बन्द … Read more   People’s News : काठमाडौं\\xa0: ‘शक्तिशाली र घातक हमला’सुरु गरेको दाबी गरेका … Read more   People’s News : २०६२/०६३ सालको दोस्रो जनआन्दोलनको सफलता भनेको सशस्त्र विद्रोहीलाई … Read more   People’s News : युग एआई (यान्त्रिक अक्कल) मय बनिसकेको छ । … Read more   नेपालमा कानुन कसैलाई पुग्ने गरी लागेको छ भने, ती हुन्– रवि … Read more   भनिन्छ देशका नेतृत्व भ्रष्टाचारमा लिप्त भए देशमा व्यापारी माफियाको रजाय हुन्छ … Read more   People’s News : ज्यान जोगाउन हजारौँ मानिसहरू आफ्ना घरगाउँ छोडेर भागेका … Read more   peoples news :\\xa0युवाहरू व्यक्ति पूजक बन्नुका पछाडि सामाजिक, राजनीतिक, मनोवैज्ञानिक र … Read more   Peopl’s News : लोकतन्त्रको सबैभन्दा ठूलो शक्ति सचेत नागरिक हुन् । … Read more \\n\\t\\t\\t\\t\\t\\t\\t\\t\\t\\t\\t\\t\\tYou must be logged in to post a comment.\\t\\t\\t\\t\\t\\t\\t\\t\\t\\t\\t\\t'}\n"
     ]
    }
   ],
   "source": [
    "url=\"https://pnews.com.np/\"\n",
    "def scrape_article(url):\n",
    "    if not is_allowed(url):\n",
    "        print(\"Scraping not allowed by robots.txt\")\n",
    "        return None\n",
    "    \n",
    "    headers = {\"User-Agent\": \"Mozilla/5.0\"}\n",
    "    response = requests.get(url, headers=headers)\n",
    "    soup = BeautifulSoup(response.text, \"html.parser\")\n",
    "\n",
    "    title = soup.find(\"h1\")\n",
    "    author = soup.find(\"span\", class_=\"author\")\n",
    "    date = soup.find(\"time\")\n",
    "    category = soup.find(\"meta\", property=\"article:section\")\n",
    "    paragraphs = soup.find_all(\"p\")\n",
    "\n",
    "    return {\n",
    "        \"title\": title.get_text(strip=True) if title else None,\n",
    "        \"author\": author.get_text(strip=True) if author else \"Unknown\",\n",
    "        \"date\": date.get_text(strip=True) if date else None,\n",
    "        \"category\": category[\"content\"] if category else None,\n",
    "        \"content\": \" \".join([p.get_text() for p in paragraphs])\n",
    "    }\n",
    "article_data = scrape_article(url)\n",
    "print(article_data)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a0bf0afd",
   "metadata": {},
   "source": [
    "20. Store the scraped news data in a structured format for analysis."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 52,
   "id": "921ed21c",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>title</th>\n",
       "      <th>author</th>\n",
       "      <th>date</th>\n",
       "      <th>category</th>\n",
       "      <th>content</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>None</td>\n",
       "      <td>पिपुल्स न्युज</td>\n",
       "      <td>February 24, 2026</td>\n",
       "      <td>None</td>\n",
       "      <td>People’s News : चुनाव आउन जम्मा नौ दिन बाँकी...</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "  title         author               date category  \\\n",
       "0  None  पिपुल्स न्युज  February 24, 2026     None   \n",
       "\n",
       "                                             content  \n",
       "0    People’s News : चुनाव आउन जम्मा नौ दिन बाँकी...  "
      ]
     },
     "execution_count": 52,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "articles = []\n",
    "articles.append(scrape_article(\"https://pnews.com.np/\"))\n",
    "\n",
    "df = pd.DataFrame(articles)\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b232521e",
   "metadata": {},
   "source": [
    "2.5.1 Handling large datasets with chunking and lazy evaluation"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a3155cc4",
   "metadata": {},
   "source": [
    "22. Process CSV files that are larger than available system memory."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "5a715a20",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "\n",
    "file_path = \"large_dataset.csv\"\n",
    "\n",
    "chunk_size = 100_000\n",
    "chunks = pd.read_csv(file_path, chunksize=chunk_size)\n",
    "\n",
    "for i, chunk in enumerate(chunks):\n",
    "    chunk[\"processed_column\"] = chunk[\"some_column\"] * 2\n",
    "    \n",
    "    chunk.to_csv(\"processed_large_dataset.csv\", mode=\"a\", index=False, header=(i==0))\n",
    "    \n",
    "    print(f\"Processed chunk {i+1}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8cec94bd",
   "metadata": {},
   "source": [
    "23. Explain how chunk-based processing works in pandas.\n",
    "\n",
    "1. Iterator-based reading: pd.read_csv(chunksize-N) returns an iterator (TextFileReader) instead of the whole DataFrame.\n",
    "2. Lazy evaluation: Only the current chunk is in memory. Old chunks can be processed and discarded.\n",
    "3. Streaming computation: You can aggregate stats across chunks without holding all data.\n",
    "Example: calculate sum of a column in a memory-efficient way:\n",
    "total = 0\n",
    "for chunk in pd.read_csv(file_path, chunksize=100_000):\n",
    "    total += chunk[\"some_column\"].sum()\n",
    "print(\"Total sum:\", total)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f3e6bb04",
   "metadata": {},
   "source": [
    "24. Monitor and limit memory usage while processing large datasets.\n",
    "->\n",
    "    1. use dtypes to reduce memory footprints:\n",
    "    df = pd.read_csv(file_path, dtype={\"id\": \"int32\", \"category\": \"category\"})\n",
    "\n",
    "    2.delete temp objects to free memory:\n",
    "    import gc\n",
    "    for chunk in pd.read_csv(file_path, chunksize=100_000):\n",
    "    # process chunk\n",
    "    del chunk\n",
    "    gc.collect()\n",
    "\n",
    "    3. monitor memory with  psutil:\n",
    "    import psutil\n",
    "    print(f\"Memory usage: {psutil.virtual_memory().percent}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e787c2a0",
   "metadata": {},
   "source": [
    "25. Optimize file I/O operations for large-scale data.\n",
    "-> 1. use binary formats like   parquet\n",
    "        # Write chunked data to Parquet\n",
    "        for i, chunk in enumerate(pd.read_csv(file_path, chunksize=100_000)):\n",
    "        chunk.to_parquet(\"processed_dataset.parquet\", engine=\"pyarrow\", compression=\"snappy\", index=False)\n",
    "    2.only read neccessary columns:\n",
    "        cols = [\"id\", \"value\", \"category\"]\n",
    "        pd.read_csv(file_path, usecols=cols, chunksize=100_000)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a0199314",
   "metadata": {},
   "source": [
    "26. Track and display progress while processing large files."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "57034755",
   "metadata": {},
   "outputs": [],
   "source": [
    "chunksize = 100_000\n",
    "reader = pd.read_csv(file_path, chunksize=chunksize)\n",
    "\n",
    "for chunk in tqdm(reader, desc=\"Processing chunks\"):\n",
    "    # Process chunk\n",
    "    chunk[\"processed_column\"] = chunk[\"some_column\"] ** 2"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1dff10c4",
   "metadata": {},
   "source": [
    "Capstone Project: Integrated Data Engineering Platform"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a41ef241",
   "metadata": {},
   "source": [
    "28. Choose a real-world domain and identify relevant data sources.\n",
    "# Domain: Real Estate Market Analysis\n",
    "\n",
    "# Relevant data sources:\n",
    "# 1. APIs: OpenWeatherMap API for weather data\n",
    "# 2. CSV files: historical housing prices\n",
    "# 3. Web scraping: property listings from a sample website"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "ebade988",
   "metadata": {},
   "source": [
    "29. Design a complete data pipeline including ingestion, storage, processing, API, and\n",
    "visualization layers.\n",
    "# Layers of the pipeline:\n",
    "\n",
    "# 1. Ingestion: APIs, CSV files, Web scraping\n",
    "# 2. Storage: SQLite database using SQLAlchemy\n",
    "# 3. Processing: pandas for cleaning and merging\n",
    "# 4. API: FastAPI to expose processed data\n",
    "# 5. Visualization: Plotly or Matplotlib"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "50fb3632",
   "metadata": {},
   "source": [
    "30. Integrate multiple data sources (APIs, files, or web scraping) into a single platform."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "599ffc6c",
   "metadata": {},
   "outputs": [],
   "source": [
    "api_url = \"https://api.openweathermap.org/data/2.5/weather\"\n",
    "params = {\"q\": \"New York\", \"appid\": \"YOUR_API_KEY\"}  # dont have a key\n",
    "weather_data = requests.get(api_url, params=params).json()\n",
    "\n",
    "weather_df = pd.DataFrame([{\n",
    "    \"city\": \"New York\",\n",
    "    \"temperature\": weather_data['main']['temp'],\n",
    "    \"weather\": weather_data['weather'][0]['description']\n",
    "}])\n",
    "\n",
    "# CSV ingestion\n",
    "historical_prices = pd.read_csv(\"historical_prices.csv\")  \n",
    "\n",
    "# Web scraping example\n",
    "from bs4 import BeautifulSoup\n",
    "\n",
    "url = \"https://example-realestate-site.com/newyork\"\n",
    "response = requests.get(url)\n",
    "soup = BeautifulSoup(response.text, \"html.parser\")\n",
    "\n",
    "listings = []\n",
    "for listing in soup.select(\".listing\"):\n",
    "    listings.append({\n",
    "        \"title\": listing.select_one(\".title\").text,\n",
    "        \"price\": float(listing.select_one(\".price\").text.replace(\"$\",\"\").replace(\",\",\"\")),\n",
    "        \"location\": listing.select_one(\".location\").text\n",
    "    })\n",
    "\n",
    "web_df = pd.DataFrame(listings)\n",
    "\n",
    "# Merge all sources\n",
    "merged_df = historical_prices.merge(weather_df, on=\"city\", how=\"left\")\n",
    "merged_df = merged_df.merge(web_df, left_on=\"city\", right_on=\"location\", how=\"left\")\n",
    "merged_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "04abe192",
   "metadata": {},
   "source": [
    "31. Store processed data efficiently using an appropriate database or storage system."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0baea311",
   "metadata": {},
   "outputs": [],
   "source": [
    "Base = declarative_base()\n",
    "\n",
    "class Property(Base):\n",
    "    __tablename__ = \"properties\"\n",
    "    id = Column(Integer, primary_key=True)\n",
    "    title = Column(String)\n",
    "    city = Column(String)\n",
    "    date = Column(Date)\n",
    "    price = Column(Float)\n",
    "    temperature = Column(Float)\n",
    "    weather = Column(String)\n",
    "\n",
    "# Create SQLite database\n",
    "engine = create_engine(\"sqlite:///realestate.db\")\n",
    "Base.metadata.create_all(engine)\n",
    "Session = sessionmaker(bind=engine)\n",
    "session = Session()\n",
    "\n",
    "# Insert processed data\n",
    "for _, row in merged_df.iterrows():\n",
    "    property_row = Property(\n",
    "        title=row.get(\"title\"),\n",
    "        city=row.get(\"city\"),\n",
    "        date=row.get(\"date\", pd.Timestamp.today()).date(),\n",
    "        price=row.get(\"price\", 0),\n",
    "        temperature=row.get(\"temperature\"),\n",
    "        weather=row.get(\"weather\")\n",
    "    )\n",
    "    session.add(property_row)\n",
    "session.commit()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "22edba25",
   "metadata": {},
   "source": [
    "32. Expose processed data through an API layer."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4e823047",
   "metadata": {},
   "outputs": [],
   "source": [
    "app = FastAPI()\n",
    "\n",
    "@app.get(\"/properties\")\n",
    "def get_properties(city: str = None):\n",
    "    session = Session(bind=engine)\n",
    "    query = session.query(Property)\n",
    "    if city:\n",
    "        query = query.filter(Property.city == city)\n",
    "    results = query.all()\n",
    "    session.close()\n",
    "    return [\n",
    "        {\n",
    "            \"id\": p.id,\n",
    "            \"title\": p.title,\n",
    "            \"city\": p.city,\n",
    "            \"date\": str(p.date),\n",
    "            \"price\": p.price,\n",
    "            \"temperature\": p.temperature,\n",
    "            \"weather\": p.weather\n",
    "        } for p in results\n",
    "    ]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "df0d7d6a",
   "metadata": {},
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "venv",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.1"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
